# Quack-Weave core and native architecture

Source snapshot `23d96d45b94c4673bf202a21718d7e6c93130117`. The native crate assembles the extension, installs DuckDB's C++ graph grammar and optimizer hooks, and implements the graph engine adapters. The core crate owns graph query binding, registration identity, scan-route selection, and the typed batch contract. DuckDB owns SQL evaluation and decides when to request graph rows.

Start with the crate topology, then follow a read from SQL lowering through a provider session. The type inventory identifies the Rust handoffs. The decision and lifecycle diagrams explain the routes, replacement races, and evidence flow from a completed scan.

The executable diagrams cover all 13 Mermaid headers with an implemented profile in the notebook's NS-Mermaid registry. Solver results apply to the declared topology, guards, traces, and numeric inputs. The source links and SQLLogic cases cover the compiled implementation. The appendix records the registry status of the other headers.

## Build and registration topology

The native `rusty_quack` crate builds a `cdylib` and exports `quack_weave` through DuckDB C API v2.0.1. Its entrypoint first calls core `register` for the portable template functions. It then creates the enabled `GraphProvider` values, calls core `register_query` with a `GqlParser` function pointer, and installs the native grammar using the returned registry. The normal build includes Ladybug, RelateDB and Neo4j. The native `template-compat` entrypoint and the WASM entrypoint call only the portable registration path.

The link dependency runs from native to core. At runtime, the core calls native engine implementations through the `ReadBackend`, `ReadSession` and `BatchReader` traits. The native C++ code also calls the registered core table functions by name. The arrows below show those runtime calls and declared zone crossings.

[Native entrypoint](../crates/native/src/lib.rs) · [Native Cargo features](../crates/native/Cargo.toml) · [Core portable registration](../crates/core/src/lib.rs) · [WASM entrypoint](../crates/wasm/src/lib.rs)

In [ ]:
architecture-beta
    group host(cloud)[DuckDB host]
    group native(cloud)[crates native]
    group core(cloud)[crates core]
    group engines(cloud)[Read engines]
    group wasm(cloud)[WASM target]
    service sql(server)[Parser Planner and Operators] in host
    service entry(server)[Native Extension Entry] in native
    service grammar(server)[C++ Grammar and Optimizer Hooks] in native
    service adapters(server)[Rust Graph Adapters] in native
    service template(server)[Portable Functions] in core
    service registry(database)[ProviderRegistry and ConfigSnapshot] in core
    service query(server)[GraphQuery and ScanProjection] in core
    service bridge(server)[Scan and Typed Batch Bridge] in core
    service lady(database)[Ladybug] in engines
    service relate(database)[RelateDB] in engines
    service neo(database)[Neo4j Query API] in engines
    service wasm_entry(server)[Rusty Quack WASM Entry] in wasm
    sql:R --> L:entry
    sql:B --> T:grammar
    entry:R --> L:template
    entry:B --> T:registry
    grammar:R --> L:query
    query:T --> B:registry
    query:R --> L:adapters
    query:B --> T:bridge
    adapters:T --> B:lady
    adapters:R --> L:relate
    adapters:B --> T:neo
    wasm_entry:R --> L:template
    %% @ns-zone-allow host native
    %% @ns-zone-allow native core
    %% @ns-zone-allow core native
    %% @ns-zone-allow native engines
    %% @ns-zone-allow wasm core

### Provider boundary and data ownership

| Provider | Evidence at bind | Native execution and delivery | Rewrite policy |
|---|---|---|---|
| Ladybug | Local observed catalog and available table populations | Read-only prepared Cypher through Ladybug FFI. The reader owns the connection and result, decodes Arrow or graph values, and constructs typed batches. | Label-catalog rewrites and the opt-in ordered runtime-key prefix. |
| RelateDB | Observed catalog when available, or a declared schema for binding | Opens a read connection for the registered bundle. The adapter calls `query_with_params` and drains its eager row result into batches. | Catalog-backed rewrites require observed types. The ordered prefix stays in DuckDB. |
| Neo4j | An explicit schema or declared catalog for the supported graph fragment | Builds a remote session, resolves an approved credential, and sends the Query API request over HTTPS. The adapter reads and decodes a capped response before yielding batches. | The descriptor disables PGQ rewrites. DuckDB can still evaluate inference-supported reads and their residual SQL. |

The local registry validates external access and the file identity. The remote path validates DuckDB's remote permission, an approved origin, a credential reference, and request limits before submission. A batch-size budget limits bridge output. Provider-internal work and eager intermediates can exceed the size of a batch.

[Ladybug adapter](../crates/native/src/ladybug.rs) · [RelateDB adapter](../crates/native/src/relate_backend.rs) · [Neo4j adapter](../crates/native/src/neo4j_backend.rs) · [Remote config and limits](../crates/core/src/qw_query/remote.rs)

In [ ]:
architecture
    group runtime(cloud)[DuckDB Process]
    group data(cloud)[Local Graph Files]
    group remote(cloud)[Remote Graph]
    service query(server)[Core Admission and Scan] in runtime
    service file_reader(server)[Native Local Adapter] in runtime
    service http_reader(server)[Native Neo4j Adapter] in runtime
    service secret(server)[DuckDB Secret Resolver] in runtime
    service lady(database)[Ladybug Database] in data
    service relate(database)[RelateDB Bundle] in data
    service aura(database)[Approved Neo4j HTTPS Origin] in remote
    query:T --> B:file_reader
    query:B --> T:http_reader
    file_reader:T --> B:lady
    file_reader:R --> L:relate
    http_reader:R --> L:aura
    http_reader:B --> T:secret
    %% @ns-zone-allow runtime data
    %% @ns-zone-allow runtime remote

## Types crossing the core/native boundary

| Contract | Types and ownership |
|---|---|
| Startup and capability injection | Native constructs `GraphProvider` from `ProviderDescriptor` and an `Arc<dyn ReadBackend>`. `ProviderSource` selects `LocalFile` or `RemoteEndpoint`. `ProviderRewrite` selects `Disabled`, `LabelCatalog`, or `ObservedCatalog`. Core returns a `ProviderRegistry` clone for the native session graph option. |
| Registration and cost evidence | `PreparedRegistration<'a>` returns `GraphCatalog`, an optional single-pattern row bound, optional `GraphPopulations`, and a publication callback. `GraphCatalog` is a map of label or kind-qualified names to typed properties. `GraphPopulations` carries by-table counts or node and edge totals. The callback may return a `RetiredRegistration` to drop outside the registry mutex. Core uses these values for schema, bind-time estimates and constrained rewrites. |
| Admission | Core binds an `Arc<ConfigSnapshot>` with a `GraphGeneration` and the selected backend. Local `open_for_generation` gets the path, schema, timeout and admitted generation. Remote `open_remote` gets the executing `Context`, `RemoteSource`, `ScanLimits` and schema. `RemoteCredential` identifies the credential reference transported to the remote adapter. |
| GQL conversion | `GqlParser` maps source text to `ParsedRead` or `ParseError`. The AST carries `NodePattern`, `Property`, `Projection`, `Predicate`, `PredicateStyle`, `Comparison` and `Literal`. Error diagnostics use `ParseErrorKind`, `Location` and `SourceSpan`. Core owns the supported conversion to Cypher. |
| Scan offers | `ReadBackend::runtime_watch_keys` returns copied `i64` keys. Core represents them as `ExactRuntimeKeys`. `OrderedWatchlistLimit` carries the graph order slot, exact join slot and positive limit. The proof comes from the native C++ optimizer and the Ladybug adapter owns the Cypher ordering suffix. |
| Read request and ownership | `ReadBackend` opens a boxed `ReadSession`. `ReadSession::execute_with_params` consumes the session and accepts Cypher and a slice of named `GraphParameter` values. Supported parameter variants are null, bool, integer, double, and text. The result is a boxed `BatchReader` or a `BackendError`. |
| Results | A `ColumnSpec` holds a name and `ScalarType`. The explicit SQL schema supports BOOLEAN, BIGINT, DOUBLE and VARCHAR. Inferred graph values also use STRUCT and LIST. A `Batch` owns `ColumnData` with a row count. A nested `GraphColumn` owns typed `GraphValue` rows. Core checks the returned schema and batch layout, then writes DuckDB `DataChunk` vectors. |
| Errors and EOF | `BatchReader::next_batch(max_rows, max_bytes)` returns `Some(Batch)` for a valid nonempty batch and `None` for verified EOF. Core translates `BackendError` to stable SQL families for open, admission, execution, schema, Arrow, timeout, batch budget, remote config, auth, transport and response budget. |

The C++ shim uses a borrowed DuckDB factory handle during installation. It copies bounded `int64_t` keys and exposes a `uint64_t` ordered limit during the synchronous `InitWatch` callback. Rust owns the copied keys and the scan-local projection after the callback returns. The C++ optimizer and grammar shim compile against DuckDB's pinned headers. 

[Core graph contracts](../crates/core/src/qw_query.rs#L62) · [Registry and provider descriptors](../crates/core/src/qw_query/providers.rs) · [Core GQL types](../crates/core/src/gql.rs) · [Typed graph values](../crates/core/src/qw_query/graph_value.rs) · [Native graph entry and FFI](../crates/native/src/graph_entry.rs) · [Build setup](../crates/native/build.rs)

## From SQL to the first provider batch

1. `GRAPH_TABLE` lowers into a DuckDB table function and a verified direct-pattern descriptor. Standalone `MATCH` lowers through the selected session graph and dialect. `qw_query` is the explicit Cypher or GQL path with a declared scalar schema and optional named bindings.
2. Core `GraphQuery::bind` resolves the provider and registration snapshot. It validates or infers the schema, binds parameters, and stores a `BoundRead`. Static predicate and projection rewrites use the verified `GraphFragment` and provider capabilities.
3. The optimizer hooks in native C++ attach an eligible runtime join offer and optional ordered-limit proof. A separate pre-optimizer rule can delegate limited ungrouped aggregates under provider and overflow guards. `EXPLAIN` describes available operations using bind information.
4. Core `init_local_state` builds a `ScanProjection`. The first `exec` pull admits the bound generation, opens the provider session, and dispatches the selected request. Later pulls reuse the same reader. Core validates each batch and writes the requested SQL column slots.

This sequence follows a successful local runtime-key read. It models one ordered trace and its declared admission premises.

[Grammar and lowering](../crates/native/grammar/standalone.cpp) · [GRAPH_TABLE](../crates/native/grammar/graph_table.cpp) · [Aggregate rule](../crates/native/grammar/aggregate_pushdown.cpp) · [GraphQuery lifecycle](../crates/core/src/qw_query.rs#L870)

In [ ]:
sequenceDiagram
    participant D as DuckDB
    participant N as Native C++
    participant C as Core GraphQuery
    participant R as Core Registry
    participant A as Native Adapter
    participant G as Graph Engine
    Note over D,G: @spec LOCAL_GRAPH_QUERY<br/>@input current_generation: Bool<br/>@input exact_offer: Bool<br/>@requires PRE: current_generation = true and exact_offer = true
    D->>N: Lower GRAPH_TABLE
    Note over D,N: @message LOWER<br/>@from D<br/>@to N<br/>@event Lower GRAPH_TABLE<br/>@order 1<br/>@when true<br/>@ensures LOWER_CONTRACT: true
    N->>C: Bind table function with PGQ fragment
    Note over N,C: @message BIND<br/>@from N<br/>@to C<br/>@event Bind table function with PGQ fragment<br/>@order 2<br/>@when true<br/>@ensures BIND_CONTRACT: true
    C->>R: Resolve alias snapshot and catalog
    Note over C,R: @message RESOLVE<br/>@from C<br/>@to R<br/>@event Resolve alias snapshot and catalog<br/>@order 3<br/>@when true<br/>@ensures RESOLVE_CONTRACT: current_generation = true
    D->>N: Initialize physical scan with join filter
    Note over D,N: @message INIT<br/>@from D<br/>@to N<br/>@event Initialize physical scan with join filter<br/>@order 4<br/>@when true<br/>@ensures INIT_CONTRACT: true
    N->>C: Expose copied exact-key offer
    Note over N,C: @message OFFER<br/>@from N<br/>@to C<br/>@event Expose copied exact-key offer<br/>@order 5<br/>@when true<br/>@ensures OFFER_CONTRACT: exact_offer = true
    D->>C: Pull graph rows
    Note over D,C: @message PULL<br/>@from D<br/>@to C<br/>@event Pull graph rows<br/>@order 6<br/>@when true<br/>@ensures PULL_CONTRACT: true
    C->>R: Check generation while opening local session
    Note over C,R: @message ADMIT<br/>@from C<br/>@to R<br/>@event Check generation while opening local session<br/>@order 7<br/>@when true<br/>@ensures ADMIT_CONTRACT: current_generation = true
    C->>A: Execute selected read with typed parameters
    Note over C,A: @message REQUEST<br/>@from C<br/>@to A<br/>@event Execute selected read with typed parameters<br/>@order 8<br/>@when true<br/>@ensures REQUEST_CONTRACT: current_generation = true
    A->>G: Invoke provider query
    Note over A,G: @message RUN<br/>@from A<br/>@to G<br/>@event Invoke provider query<br/>@order 9<br/>@when true<br/>@ensures RUN_CONTRACT: true
    A->>C: Return schema and typed batch
    Note over A,C: @message BATCH<br/>@from A<br/>@to C<br/>@event Return schema and typed batch<br/>@order 10<br/>@when true<br/>@ensures BATCH_CONTRACT: true
    C->>D: Write DataChunk for residual SQL
    Note over C,D: @message WRITE<br/>@from C<br/>@to D<br/>@event Write DataChunk for residual SQL<br/>@order 11<br/>@when true<br/>@ensures WRITE_CONTRACT: true
    Note over D,G: @verify LOCAL_PROTOCOL: prove sequence_protocol

### Route selection belongs to scan initialization

The `qw_enable_runtime_watchlist` session switch gates both literal and runtime exact offers. A verified literal candidate takes precedence. The regular runtime branch needs a typed exact key offer. Missing, inexact, or disabled offers keep the host route. Static predicate rewrites have their own eligibility checks. The SQL join still determines the result bag, including duplicate keys and NULLs.

The provider-bound prefix requires `qw_enable_ordered_watchlist_limit`, an eligible Ladybug runtime-key join, the exact join slot, one visible direct BIGINT order value, ASC NULLS LAST, and a positive limit with zero offset. The eligible plan has no additional graph-side filters. DuckDB executes the final join and top-k. Since each matching graph row contributes at least one SQL row and all projected ties carry the same value, the first k qualifying graph rows suffice for this shape. Ladybug supplies the explicit null-last sort expression. RelateDB and literal-key joins use the applicable incumbent route for top-k.

`QuackWeaveWatchlist` logs the exact-key offer or its decline. `QuackWeaveGraphAdmission` reports the acquired generation. `QuackWeaveGraphRoute` logs the selected route once execution returns a reader. An optional prefilter label in `EXPLAIN` describes the available plan and the route log records the dispatch.

[Core route gate](../crates/core/src/qw_query/projection.rs) · [Runtime rules and logging](../crates/core/src/qw_query/runtime_watch.rs) · [Optimizer offer](../crates/native/grammar/runtime_watchlist.cpp) · [Ordered shape proof](../crates/native/grammar/ordered_watchlist.hpp)

In [ ]:
flowchart TD
    S["`@spec ROUTE_AT_SCAN_INIT
@type Status = enum[host,literal_exact,runtime_exact,bounded_exact]
@input enabled: Bool
@input literal_valid: Bool
@input runtime_valid: Bool
@input bound_eligible: Bool
@input bound_enabled: Bool
@input host_sample: Bool
@output status: Status
@requires PRE: true`"]
    B0["`@branch DISPATCH_HOST
@when enabled = false or (literal_valid = false and (runtime_valid = false or host_sample = true))
@ensures DISPATCH_HOST_RESULT: status = host`"]
    B1["`@branch USE_LITERAL
@when enabled = true and literal_valid = true
@ensures USE_LITERAL_RESULT: status = literal_exact`"]
    B2["`@branch USE_BOUND
@when enabled = true and literal_valid = false and runtime_valid = true and host_sample = false and bound_eligible = true and bound_enabled = true
@ensures USE_BOUND_RESULT: status = bounded_exact`"]
    B3["`@branch USE_RUNTIME
@when enabled = true and literal_valid = false and runtime_valid = true and host_sample = false and (bound_eligible = false or bound_enabled = false)
@ensures USE_RUNTIME_RESULT: status = runtime_exact`"]
    V["`@verify COVERED: prove partition_coverage
@verify DISJOINT: prove partition_exclusive
@verify DETERMINISTIC: prove determinism
@verify STATUS_WITNESSES: witness each status`"]
    S --> B0 --> V
    S --> B1 --> V
    S --> B2 --> V
    S --> B3 --> V

## Generation and reader lifecycle

Core publishes a local registration and its provider cache owner under a per-provider write gate. It checks the bound generation and acquires the native session under the matching read gate. Host configuration and logging callbacks stay outside that gate. A replacement that wins before admission causes a stale bound plan to fail with `QW_CONFIG`. An already admitted reader owns the original session resources and can finish after replacement. A freshly rebound query resolves the new registration. Remote read admission uses a final generation check before submission, after no-I/O session construction.

The first state diagram models the scan cursor. `Dormant` obtains a reader on the first pull. A valid batch keeps it `Active`. Verified EOF or an error releases the reader and finishes the scan. The second state diagram models the local replacement schedules and the invariant that an open session stays pinned to its bound generation.

[Generation gate](../crates/core/src/qw_query/providers.rs#L279) · [Scan state and exec](../crates/core/src/qw_query.rs#L567) · [SQLLogic replacement test](../test/stabilization/sqllogic/usage_binding.test)

In [ ]:
stateDiagram-v2
    [*] --> Dormant
    Dormant --> Active: first_batch
    Dormant --> Finished: empty_eof
    Dormant --> Finished: open_error
    Active --> Active: next_batch
    Active --> Finished: verified_eof
    Active --> Finished: read_error
    note right of Dormant
      @spec SCAN_READER_LIFETIME
      @type Event = enum[first_batch,empty_eof,open_error,next_batch,verified_eof,read_error]
      @input event: Event
      @state-var owns_reader: Bool
      @state-var has_batch: Bool
      @requires PRE: owns_reader = false and has_batch = false
      @state Dormant
      @invariant BATCH_REQUIRES_READER: has_batch = false or owns_reader = true
    end note
    note right of Active
      @state Active
    end note
    note right of Finished
      @state Finished
    end note
    note right of Active
      @transition START
      @from Dormant
      @to Active
      @event event = first_batch
      @guard owns_reader = false
      @update owns_reader' = true
      @update has_batch' = true
    end note
    note right of Finished
      @transition EMPTY
      @from Dormant
      @to Finished
      @event event = empty_eof
      @guard owns_reader = false
      @update owns_reader' = false
      @update has_batch' = false
    end note
    note right of Finished
      @transition START_FAIL
      @from Dormant
      @to Finished
      @event event = open_error
      @guard owns_reader = false
      @update owns_reader' = false
      @update has_batch' = false
    end note
    note right of Active
      @transition NEXT
      @from Active
      @to Active
      @event event = next_batch
      @guard owns_reader = true
      @update has_batch' = true
    end note
    note right of Finished
      @transition END
      @from Active
      @to Finished
      @event event = verified_eof
      @guard owns_reader = true
      @update owns_reader' = false
      @update has_batch' = false
    end note
    note right of Finished
      @transition FAIL
      @from Active
      @to Finished
      @event event = read_error
      @guard owns_reader = true
      @update owns_reader' = false
      @update has_batch' = false
    end note
    note right of Dormant
      @verify INIT_BATCH_REQUIRES_READER: prove initiate BATCH_REQUIRES_READER
      @verify BATCH_REQUIRES_READER_START: prove preserve BATCH_REQUIRES_READER on START
      @verify BATCH_REQUIRES_READER_EMPTY: prove preserve BATCH_REQUIRES_READER on EMPTY
      @verify BATCH_REQUIRES_READER_START_FAIL: prove preserve BATCH_REQUIRES_READER on START_FAIL
      @verify BATCH_REQUIRES_READER_NEXT: prove preserve BATCH_REQUIRES_READER on NEXT
      @verify BATCH_REQUIRES_READER_END: prove preserve BATCH_REQUIRES_READER on END
      @verify BATCH_REQUIRES_READER_FAIL: prove preserve BATCH_REQUIRES_READER on FAIL
    end note

In [ ]:
stateDiagram
    [*] --> Bound
    Bound --> Open: open_current
    Bound --> Stale: replace_before
    Open --> Superseded: replace_after
    Stale --> Done: reject
    Superseded --> Done: drain
    note right of Bound
      @spec LOCAL_GENERATION_ADMISSION
      @type Event = enum[open_current,replace_before,replace_after,reject,drain]
      @input event: Event
      @state-var current: Int
      @state-var bound: Int
      @state-var pinned: Int
      @state-var opened: Bool
      @requires PRE: current = bound and pinned = bound and opened = false
      @state Bound
      @invariant READER_PINS_BOUND: opened = false or pinned = bound
    end note
    note right of Open
      @state Open
    end note
    note right of Stale
      @state Stale
    end note
    note right of Superseded
      @state Superseded
    end note
    note right of Done
      @state Done
    end note
    note right of Open
      @transition ADMIT
      @from Bound
      @to Open
      @event event = open_current
      @guard current = bound and opened = false
      @update opened' = true
      @update pinned' = current
    end note
    note right of Stale
      @transition BEFORE
      @from Bound
      @to Stale
      @event event = replace_before
      @guard current = bound and opened = false
      @update current' = current + 1
    end note
    note right of Superseded
      @transition AFTER
      @from Open
      @to Superseded
      @event event = replace_after
      @guard current = bound and opened = true
      @update current' = current + 1
    end note
    note right of Done
      @transition REJECT
      @from Stale
      @to Done
      @event event = reject
      @guard current > bound and opened = false
      @update opened' = false
    end note
    note right of Done
      @transition DRAIN
      @from Superseded
      @to Done
      @event event = drain
      @guard current > bound and opened = true
      @update opened' = false
    end note
    note right of Bound
      @verify INIT_READER_PINS_BOUND: prove initiate READER_PINS_BOUND
      @verify READER_PINS_BOUND_ADMIT: prove preserve READER_PINS_BOUND on ADMIT
      @verify READER_PINS_BOUND_BEFORE: prove preserve READER_PINS_BOUND on BEFORE
      @verify READER_PINS_BOUND_AFTER: prove preserve READER_PINS_BOUND on AFTER
      @verify READER_PINS_BOUND_REJECT: prove preserve READER_PINS_BOUND on REJECT
      @verify READER_PINS_BOUND_DRAIN: prove preserve READER_PINS_BOUND on DRAIN
    end note

### Dependencies along the read and registration paths

These schedules use one artificial slot per dependency on an arbitrary reference date. They show prerequisites for one read and one registration. Actual timing and overlap come from the runtime and independent provider gates.

In [ ]:
gantt
    title Local read prerequisites on a logical clock
    dateFormat YYYY-MM-DD
    axisFormat %d
    section Dependency order
    Bind snapshot :bind, 2000-01-01, 1d
    Acquire session under admission gate :admit, after bind, 1d
    Submit selected Cypher :request, after admit, 1d
    Validate and publish typed batch :batch, after request, 1d

In [ ]:
timeline
    title Registration ordering with logical slots
    Read registration inputs :prepare, 2000-01-01, 1d
    Validate file identity and descriptor :validate, after prepare, 1d
    Publish registration and backend owner :publish, after validate, 1d
    Admit a bound reader :admit, after publish, 1d

## Evidence flows back into the registration snapshot

`qw_graph_route_feedback` defaults to `off`. A structured runtime-key scan can collect a sample in `shadow` or explicitly run a broad `host_sample` for comparison. Core times active provider and bridge pulls. At verified EOF it checks that the generation is current before publishing. Failed or partial reads produce no sample.

The cache holds one shape per registration, keyed by the base graph query, projected slots, exact key slot and normalized key set, and the verified ordered shape when present. It compares the latest completed exact, host, or bounded samples for that shape. A new registration gets a fresh cache. The shadow output is a comparison for the operator to inspect. Runtime selection continues to follow the route and capability gates above.

[Feedback cache](../crates/core/src/qw_query/route_feedback.rs) · [Scan integration](../crates/core/src/qw_query.rs#L893) · [Feedback SQLLogic](../test/relate/graph_route_feedback.test)

In [ ]:
graph TD
    S["`@spec COMPLETED_SCAN_FEEDBACK
@type Mode = enum[off,shadow,host_sample]
@type Status = enum[ignore,sample_only,report_comparison]
@input mode: Mode
@input has_probe: Bool
@input verified_eof: Bool
@input generation_current: Bool
@input paired_same_shape: Bool
@output status: Status
@requires PRE: true`"]
    B0["`@branch IGNORE
@when mode = off or has_probe = false or verified_eof = false or generation_current = false
@ensures IGNORE_RESULT: status = ignore`"]
    B1["`@branch RECORD
@when mode != off and has_probe = true and verified_eof = true and generation_current = true and (mode = host_sample or paired_same_shape = false)
@ensures RECORD_RESULT: status = sample_only`"]
    B2["`@branch COMPARE
@when mode = shadow and has_probe = true and verified_eof = true and generation_current = true and paired_same_shape = true
@ensures COMPARE_RESULT: status = report_comparison`"]
    V["`@verify COVERED: prove partition_coverage
@verify DISJOINT: prove partition_exclusive
@verify DETERMINISTIC: prove determinism
@verify STATUS_WITNESSES: witness each status`"]
    S --> B0 --> V
    S --> B1 --> V
    S --> B2 --> V

## Observable execution evidence

Run the Arrow input cell below before rerunning the four chart proofs. The cell publishes the fixed observations and the diagrams bind their schema and data hashes.

The two Sankey paths count graph rows delivered before the DuckDB join. They represent separate runs of the local LSQB hot top-16 query with default DuckDB optimizers and the same SQL result. The diagram stops at the scan boundary because a residual join can multiply rows. The two-provider latency chart shows medians from 12 paired runs of the same local query, with ordered limit off and on. The setting changes the Ladybug path on this shape. RelateDB uses its exact route in both arms.

The provider pie counts three adapters in the default native build. The verification chart counts cases in the final core Rust, provider SQLLogic, and stabilization runs. The last two suites passed 655 and 1,183 assertions. These counts and the LSQB numbers describe the local artifact built for `23d96d4`.

[Combined experiment notebook](../quack-weave-core-combined-benchmark.ipynb) · [Ordered-limit SQLLogic](../test/relate/ordered_watchlist_limit.test) · [Shared route SQLLogic](../test/relate/watchlist_applied_route.test)

In [2]:
import pyarrow as pa

# Fixed observations at the documented source snapshot. Values are integer
# ticks recorded as decimal text for the NS-Mermaid Arrow binding.
datasets = [
    {
        "port": "scan_delivery",
        "cols": [
            "source",
            "target",
            "value"
        ],
        "unit": "1",
        "rows": [
            [
                "Exact native read",
                "Exact typed batches",
                "2631"
            ],
            [
                "Exact typed batches",
                "Exact DuckDB scan",
                "2631"
            ],
            [
                "Bounded native read",
                "Bounded typed batches",
                "16"
            ],
            [
                "Bounded typed batches",
                "Bounded DuckDB scan",
                "16"
            ]
        ]
    },
    {
        "port": "lsqb_hot_latency",
        "cols": [
            "category",
            "value"
        ],
        "unit": "1",
        "rows": [
            [
                "Ladybug off",
                "2122"
            ],
            [
                "Ladybug on",
                "1874"
            ],
            [
                "RelateDB off",
                "3560"
            ],
            [
                "RelateDB on",
                "3574"
            ]
        ]
    },
    {
        "port": "provider_inventory",
        "cols": [
            "label",
            "value"
        ],
        "unit": "1",
        "rows": [
            [
                "Local files",
                "2"
            ],
            [
                "Remote HTTPS",
                "1"
            ]
        ]
    },
    {
        "port": "verification_cases",
        "cols": [
            "category",
            "value"
        ],
        "unit": "1",
        "rows": [
            [
                "Core Rust",
                "10"
            ],
            [
                "Provider SQLLogic",
                "22"
            ],
            [
                "Stabilization",
                "30"
            ]
        ]
    }
]

for dataset in datasets:
    schema = pa.schema([
        pa.field(name, pa.string(), nullable=False,
                 metadata={b"unit": dataset["unit"].encode()} if name == "value" else None)
        for name in dataset["cols"]
    ])
    columns = [pa.array(values, type=pa.string()) for values in zip(*dataset["rows"])]
    spur.put(dataset["port"], pa.Table.from_arrays(columns, schema=schema))


source,target,value
Exact native read,Exact typed batches,2631
Exact typed batches,Exact DuckDB scan,2631
Bounded native read,Bounded typed batches,16
Bounded typed batches,Bounded DuckDB scan,16


category,value
Ladybug off,2122
Ladybug on,1874
RelateDB off,3560
RelateDB on,3574


label,value
Local files,2
Remote HTTPS,1


category,value
Core Rust,10
Provider SQLLogic,22
Stabilization,30


In [ ]:
sankey
    %% @ns-bind producer=149feaa5-cbf2-406a-99b0-6b7484b6d5c9 port=scan_delivery snapshot=7cc2efcce05432d1fc3b3c8b9d8512cc0d75a862576d20e417dbd028b092794d schema=44f729ca7d2a5dc225e92c5d599811db8260be3ae36f104f056c5f54d935e768 unit=1
    Exact native read,Exact typed batches,2631
    Exact typed batches,Exact DuckDB scan,2631
    Bounded native read,Bounded typed batches,16
    Bounded typed batches,Bounded DuckDB scan,16

In [ ]:
xychart
    %% @ns-bind producer=149feaa5-cbf2-406a-99b0-6b7484b6d5c9 port=lsqb_hot_latency snapshot=50c238b97f9ab55c336ab133b12a25c7c7886a440cfe2d27628b8cfd47950729 schema=f7188d1799d47517ce745717ab86a9aa97dbf6c7375e3bca1198b649a90543a8 unit=1
    title "LSQB hot top-16 paired replay (median microseconds)"
    x-axis ["Ladybug off", "Ladybug on", "RelateDB off", "RelateDB on"]
    y-axis "Microseconds"
    bar [2122, 1874, 3560, 3574]

In [ ]:
pie title Native provider adapters by source
    %% @ns-bind producer=149feaa5-cbf2-406a-99b0-6b7484b6d5c9 port=provider_inventory snapshot=2a81ac26893086ef6516d7cecd4f87598ffbfa8195b89180c25459e99f63b17a schema=8ee936a11a67759da60dd14c7f971b7ed31831f78b26bbf77ef932016a3f4edd unit=1
    "Local files" : 2
    "Remote HTTPS" : 1
    %% @ns-total 3

In [ ]:
xychart-beta
    %% @ns-bind producer=149feaa5-cbf2-406a-99b0-6b7484b6d5c9 port=verification_cases snapshot=d34c2a8617e4c8cd168c439dde07640f36cadb61635a02b213f81c2f6d07f60e schema=f7188d1799d47517ce745717ab86a9aa97dbf6c7375e3bca1198b649a90543a8 unit=1
    title "Verification suites at 23d96d4"
    x-axis ["Core Rust", "Provider SQLLogic", "Stabilization"]
    y-axis "Test cases"
    bar [10, 22, 30]

## Source map by component

| Component | Types and records to inspect |
|---|---|
| Core request and result | `GraphQuery`, `GraphQueryKind`, `BoundRead`, `GraphLanguage`, `GraphFragment`, `ScanProjection`, `Scan`, `ColumnSpec`, `ScalarType`, `Batch`, `ColumnData`, `GraphColumn` and `GraphValue`. |
| Core registration and policy | `ProviderRegistry`, `RegistryState`, `ProviderSlot`, `GraphConfig`, `ConfigSnapshot`, `RegisteredConfig`, `GraphGeneration`, `RuntimeWatch`, `ExactRuntimeKeys`, `WatchlistRoute`, and feedback `Mode`, `Shape`, `Observation` and `Feedback`. |
| Native C++ | `GraphGrammar` and `GraphConnections` install and activate syntax. `WatchInit`, `KeyOffer`, `KeyCopy` and `QwOrderedWatchlistOffer` hold the temporary plan and scan offers. The aggregate and explain hooks consume graph bind information. |
| Native Rust Ladybug | `LadybugBackend`, `DatabaseCache`, `Session` and `Reader`. Native resources are wrapped by `Database`, `Connection`, `PreparedStatement` and `QueryResult`. |
| Native Rust RelateDB | `RelateBackend`, `RegisteredDatabase`, `Session` and `Rows`. The adapter owns observed type checks and conversion of the eager result. |
| Native Rust Neo4j | `Neo4jBackend`, `Endpoint`, `Session` and the decoded row reader. The secret bridge and adapter own scoped authentication and transport. |

Source navigation for an adapter addition starts at the descriptor in the native entrypoint, follows the injected traits in core, and checks the C++ rules only for features that require a DuckDB plan proof.

[Core source](../crates/core/src) · [Native Rust source](../crates/native/src) · [Native C++ grammar](../crates/native/grammar)

## NS-Mermaid profile coverage

| Registry profile | Mermaid headers | Evidence in this notebook |
|---|---|---|
| Architecture policy | `architecture-beta`, `architecture` | Runtime component and deployment edges |
| Branch relation | `flowchart`, `graph` | Route choice and feedback eligibility |
| State invariant | `stateDiagram-v2`, `stateDiagram` | Cursor and local generation invariants |
| Sequence trace | `sequenceDiagram` | Declared successful local graph read |
| Schedule feasibility | `gantt`, `timeline` | Modeled read and registration prerequisite order |
| Flow conservation | `sankey` | Counts delivered at the graph scan boundary |
| Numeric series | `pie`, `xychart`, `xychart-beta` | Provider inventory, observed latency and verification counts |
| Other numeric headers | `radar`, `quadrant`, `treemap` | Registry reports their series adapters as unavailable |
| Finite sets, workflow, and wire layout | `venn`, `kanban`, `packet`, `packet-beta` | Registry reports finite-set, workflow, and packet lowering as unavailable |
| Bounded relations and Git policy | `classDiagram`, `erDiagram`, `requirementDiagram`, `gitGraph` | Mermaid visual references below |
| Conceptual diagrams | `mindmap`, `journey`, `ishikawa`, `wardley`, `cynefin` | Registry classifies these as visualization-only |

The registry also lists bounded reachability, transition BMC, packet encoding, event trace, and schedule optimization without active Mermaid headers. The executable cells use version 1 of the installed profiles. Mermaid fences in the appendix are visual references for a compatible renderer.

### Rust trait and data model

```mermaid
classDiagram
    class GraphProvider
    class ProviderDescriptor
    class ReadBackend {
        <<interface>>
        prepare_registration()
        open_for_generation()
        open_remote()
    }
    class ReadSession {
        <<interface>>
        execute_with_params()
    }
    class BatchReader {
        <<interface>>
        schema()
        next_batch()
    }
    class GraphParameter
    class OrderedWatchlistLimit
    class Batch {
        len
        columns
    }
    class ColumnSpec
    class ScalarType
    class ColumnData
    class GraphColumn
    class GraphValue
    class LadybugBackend
    class RelateBackend
    class Neo4jBackend
    GraphProvider *-- ProviderDescriptor
    GraphProvider o-- ReadBackend
    LadybugBackend ..|> ReadBackend
    RelateBackend ..|> ReadBackend
    Neo4jBackend ..|> ReadBackend
    ReadBackend ..> ReadSession : opens
    ReadBackend ..> OrderedWatchlistLimit : offers
    ReadSession ..> GraphParameter : consumes named values
    ReadSession ..> BatchReader : transfers read ownership
    BatchReader ..> Batch : yields
    BatchReader ..> ColumnSpec : provides schema
    ColumnSpec *-- ScalarType
    Batch *-- ColumnData
    ColumnData o-- GraphColumn
    GraphColumn o-- ScalarType
    GraphColumn *-- GraphValue
```

### Ownership and requirements

The entity view describes in-memory ownership. It groups one active registration per provider together with the bound plans and feedback it can serve.

```mermaid
erDiagram
    PROVIDER_SLOT ||--o| CONFIG_SNAPSHOT : publishes
    CONFIG_SNAPSHOT ||--|| GRAPH_GENERATION : identifies
    CONFIG_SNAPSHOT ||--o{ BOUND_READ : pins
    CONFIG_SNAPSHOT ||--o| FEEDBACK_ENTRY : caches
    BOUND_READ ||--o{ SCAN_LOCAL_STATE : initializes
    SCAN_LOCAL_STATE ||--|| SCAN_PROJECTION : maps
    SCAN_LOCAL_STATE ||--|| SCAN : tracks
    SCAN ||--o| BATCH_READER : retains
    BATCH_READER ||--o{ TYPED_BATCH : yields
```

```mermaid
requirementDiagram
    requirement same_sql_bag {
        id: 1
        text: "Exact-key routes preserve the SQL result bag"
        risk: high
        verifymethod: test
    }
    requirement admitted_generation {
        id: 2
        text: "An admitted read stays on its bound generation"
        risk: high
        verifymethod: test
    }
    element watchlist_logic {
        type: SQLLogic
    }
    element usage_binding {
        type: SQLLogic
    }
    watchlist_logic - verifies -> same_sql_bag
    usage_binding - verifies -> admitted_generation
```

### Implementation history

```mermaid
gitGraph
    commit id: "54b2db6" tag: "baseline"
    commit id: "29b024c" tag: "shared route gate"
    commit id: "2d3cfec" tag: "generation admission"
    commit id: "c2356f3" tag: "ordered Ladybug prefix"
    commit id: "23d96d4" tag: "shadow feedback"
```

```mermaid
mindmap
    root((Core native handoff))
        Enter through DuckDB
            Native grammar
            Core table function
        Open a graph
            Registry generation
            Native adapter session
        Return typed rows
            BatchReader
            DuckDB DataChunk
        Observe execution
            Offer and dispatch logs
            Completed scan feedback
```